# 06. Улучшение модели Semantic News Novelty (v3)

Цель ноутбука — составить логичную цепочку экспериментов без дублирования большого служебного кода, который был сделан на предыдущих уроках (в предыдущих ноутбуках).
Общая схема всего пайплайна сервиса на текущий момент (до оптимизаций):
1. Подготовленные данные проходят через векторизатор. На текущий момент это `bge-m3`: он покрывает большую часть новостей по длине, кроме нескольких слишком длинных выбросов. Более простые векторизаторы не справлялись с таким количеством токенов: пришлось бы дробить текст, чего хотелось избежать.
2. Новости кластеризуются по близости сюжетов через графовую кластеризацию на основе семантической близости и временного окна в 14 дней.
3. Далее на данных с кластерами уже была обучена CatBoost-модель. Она училась на silver set, размеченном LLM, и валидировалась на golden set с ручной разметкой. Разметка отвечала за то, насколько новость привносит новое в сюжет из нескольких новостей.
4. Там же, в `05_semantic_baseline_model.ipynb`, был проведён анализ ошибок и модель была доработана: при разметке новых элементов кластера стал учитываться контекст из похожих кластеров прошлого. Это нужно, чтобы даже при разделении похожих сюжетов на разные кластеры контекст учитывался при принятии решения об `is_significant`.

В сумме это позволило добиться приемлемых характеристик финальных меток. В этом ноутбуке мы попытаемся улучшить данный пайплайн.

Порядок работы:

1. Воспроизводим "нулевой эксперимент" (`exp_00b`) — модель, выработанную в ноутбуке `05_semantic_baseline_model.ipynb`. Эта модель уже содержит существенные улучшения по сравнению с исходным baseline. Её используем как точку отсчёта для дальнейших улучшений. В целом та модель уже достигла целевых показателей, но есть куда расти.

2. Далее проводим эксперименты по улучшению алгоритма кластеризации. Базовая кластеризация имеет неплохой F1 на golden-разметке и почти нулевой false merge rate, но при этом чрезмерно фрагментирует сюжеты. Значит, нужно бороться с фрагментацией, не допуская большого роста ошибочных слияний. Более расширенные эксперименты можно посмотреть в ноутбуке `06_model_improvement_experiments.ipynb`; здесь приведены наиболее важные эксперименты, приведшие к результату.
3. Фиксируем наиболее эффективную кластеризацию
4. На выбранной кластеризации проводим несколько экспериментов по улучшению шага классификации.
5. Опционально проверяем на получившемся pipeline модель из ноутбука `07_...`: векторизатор, дообученный на русских парафразах триплетами.
6. Сохраняем выбранную финальную модель для `final_pipeline`.

Что закрывает требования домашней работы:

- preprocessing / feature engineering / data generation: разделы 2, 5 и 6 описывают очистку входных данных, удаление утечек, генерацию пар кандидатов, эмбеддингов и признаков;
- улучшенная архитектура модели: разделы 5-7 показывают ступенчатую архитектуру `vectorization -> clustering -> novelty classifier` и сравнение CatBoost / MLP / LogisticRegression;
- постобработка предсказаний: раздел 7 фиксирует threshold, duplicate rule, first-item fallback, review margin и экспорт runtime config;
- анализ качества: раздел 9 сравнивает baseline и финальное решение по clustering metrics, binary novelty metrics и error counts.

Связь с предыдущими ноутбуками:

- `03_data_analysis_and_dataset_preparation.ipynb`: первичный EDA, очистка Lenta.ru, формирование clean dataset, golden candidate set и шаблонов разметки;
- `04_evaluate_annotations_and_predictions.ipynb`: единые evaluation-функции, coverage checks, label distribution, pairwise clustering metrics и error analysis;
- `05_semantic_baseline_model.ipynb`: semantic baseline, inline evaluation, supervised significance model и previous-only feature engineering без утечки из будущего;
- `06_model_improvement_experiments.ipynb`: ранние improvement experiments по embeddings, clustering, centroid/top-k признакам, MLP final-step модели и анализу ошибок.

Этот v3-ноутбук является аккуратным обобщением экспериментов, проведённых в прошлых ноутбуках, для сдачи проекта: он не повторяет весь EDA из 03-05, а переиспользует подготовленные артефакты и фиксирует финальную воспроизводимую цепочку экспериментов.

## 1. Импорты и настройка путей

В ноутбуке остаётся только orchestration-код. Основная логика вынесена в пакет `model/`, `final_pipeline`.

In [26]:
import sys
from pathlib import Path

import joblib
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "model").exists():
            return candidate
    raise RuntimeError(f"Project root with src/model was not found from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"

for path in [PROJECT_ROOT, SRC_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

In [27]:
%load_ext autoreload
%autoreload 2

from final_pipeline import FinalPipelineConfig
from model.attach_clustering import (
    AttachClusteringConfig,
    BaselineClusteringConfig,
    SilverPositiveSelectionConfig,
    build_best_candidate_attach_clusters,
    build_candidate_pairs,
    evaluate_cluster_ids_on_reference,
    get_attach_config_from_sweep_row,
    make_clustered_news,
    prepare_silver_positive_reference,
    run_silver_positive_attach_sweep,
    select_silver_positive_variant,
)
from model.classifier_training import (
    fit_catboost_binary,
    fit_logreg_binary,
    fit_mlp_binary,
    make_significance_training_frame,
    train_validation_split,
)
from model.config import ExperimentPaths, SemanticNewsBaselineConfig, SignificanceModelConfig
from model.data import (
    load_annotation,
    load_clean_news,
    normalize_news_id,
    remove_train_eval_leakage,
)
from model.embeddings import SentenceTransformerEncoder
from model.evaluation import evaluate_predictions
from model.experiment_tracking import ExperimentTracker
from model.features import LEGACY_SIGNIFICANCE_FEATURE_COLUMNS, build_legacy_significance_features
from model.legacy_pipeline import LegacyNewsNoveltyPipeline, LegacyNewsPipelineConfig
from model.significance_model import CatBoostSignificanceModel

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
paths = ExperimentPaths(project_root=PROJECT_ROOT)
paths.predictions_dir.mkdir(parents=True, exist_ok=True)
paths.artifacts_dir.mkdir(parents=True, exist_ok=True)
paths.models_dir.mkdir(parents=True, exist_ok=True)

paths.clean_news_path, paths.golden_path, paths.silver_path

(WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_clean_news.csv'),
 WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_golden_set_human.csv'),
 WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_silver_set_llm.csv'))

## 2. Загрузка данных

`candidate_news` — полный набор, на котором строится кластеризация. 
`Golden` - вручную размеченный набор, используется для оценки, но не для выбора удачного эксперимента `exp_10`. 
`Silver` - набор, размеченный LLM, используется для обучения, но не финальной валидации

Preprocessing и подготовка данных в этом ноутбуке:

- `load_clean_news` приводит исходные новости к единой схеме, нормализует даты, id, темы и текстовые поля;
- `lenta_golden_candidate_pool.csv` ограничивает эксперимент воспроизводимым candidate pool, чтобы все варианты сравнивались на одном наборе объектов;
- `remove_train_eval_leakage` удаляет golden-строки из silver перед обучением и подбором параметров, поэтому golden остаётся только holdout-оценкой;
- `normalize_news_id` далее используется перед merge признаков и разметки, чтобы не терять строки из-за несовпадения типов id.

In [29]:
clean_news = load_clean_news(paths.clean_news_path)
golden = load_annotation(paths.golden_path)
silver_raw = load_annotation(paths.silver_path)
silver = remove_train_eval_leakage(silver_raw, golden, id_column="news_id")

candidate_pool_path = paths.prepared_dir / "lenta_golden_candidate_pool.csv"
if not candidate_pool_path.exists():
    raise FileNotFoundError(f"Не найден candidate pool: {candidate_pool_path}")

raw_candidate_pool = pd.read_csv(candidate_pool_path)

print("clean_news:", clean_news.shape)
print("candidate_pool:", raw_candidate_pool.shape)
print("golden:", golden.shape)
print("silver без golden leakage:", silver.shape)
print("golden clusters:", golden["cluster_id"].nunique())
print("silver clusters:", silver["cluster_id"].nunique())

clean_news: (99555, 14)
candidate_pool: (3176, 16)
golden: (121, 10)
silver без golden leakage: (1915, 10)
golden clusters: 34
silver clusters: 1736


## 3. Embeddings и сохранённая baseline-модель

Baseline использует `BAAI/bge-m3` и сохранённую CatBoost/fallback модель из предыдущего этапа.

In [30]:
baseline_config = SemanticNewsBaselineConfig()

encoder = SentenceTransformerEncoder(
    model_name=baseline_config.embedding_model_name,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

current_model = CatBoostSignificanceModel.load(
    model_path=paths.current_catboost_model_path,
    config_path=paths.current_catboost_config_path,
)

# Приводим wrapper к набору фичей, на котором считались предыдущие метрики.
current_model.config = SignificanceModelConfig(
    threshold=0.42,
    duplicate_threshold=0.90,
    review_margin=0.10,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
)
current_model.feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

tracker = ExperimentTracker(predictions_dir=paths.predictions_dir)
print(type(current_model.model))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Ignored unknown config keys: ['batch_size', 'depth', 'embeddings_cache_path', 'fallback_max_previous_candidates', 'fallback_similarity_threshold', 'fallback_window_days', 'first_item_fallback_enabled', 'iterations', 'l2_leaf_reg', 'learning_rate', 'local_model_dir', 'model_name', 'model_path', 'validation_size']
<class 'catboost.core.CatBoostClassifier'>


## 4. Baseline `exp_00b`

Воспроизводим референсный пайплайн: подготовка данных → BGE-M3 → строгая кластеризация по графу → текущая novelty model.

In [31]:
legacy_pipeline = LegacyNewsNoveltyPipeline(
    encoder=encoder,
    novelty_model=current_model,
    config=LegacyNewsPipelineConfig(
        embeddings_cache_path=(
            paths.artifacts_dir / "embeddings" / "bge_m3_candidate_pool_id_aligned.npz"
        ),
        story_threshold=0.82,
        story_window_days=14,
        use_topic_blocking=True,
        normalize_embeddings_for_clustering=True,
    ),
)

legacy_result = legacy_pipeline.predict(raw_candidate_pool)

pred_00b = legacy_result.predictions
old_pipeline_clustered_news = legacy_result.clustered_news.reset_index(drop=True)
old_pipeline_embeddings = legacy_result.embeddings
baseline_cluster_ids = old_pipeline_clustered_news["cluster_id"].astype(str).reset_index(drop=True)

metrics_00b = tracker.register(
    experiment="exp_00b_reproduced_old_pipeline",
    golden=golden,
    prediction=pred_00b,
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="reference threshold graph: threshold=0.82, window=14",
    novelty_variant="current CatBoost/fallback model",
    comment="Воспроизводимый референс из предыдущего этапа",
)

tracker.compact()

Рёбер в графе похожести: 914
Количество кластеров: 2607


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.0,0.724138,0.835616,0.835616,0.835616,Воспроизводимый референс из предыдущего этапа


## 5. Выбор более оптимальной кластеризации без классификации на последнем шаге

Сначала сравниваем только качество кластеризации. Это быстрее и логичнее: не нужно запускать финальный шаг, достаточно добиться лучшей метрики кластеризации.

In [32]:
candidate_pairs = build_candidate_pairs(
    old_pipeline_clustered_news,
    old_pipeline_embeddings,
    min_similarity=0.75,
    max_days=14,
)

print("Набор данных для обучения:", candidate_pairs.shape)

Набор данных для обучения: (2648, 10)


### 5.1. Эксперимент "с утечкой данных" `exp_09b`

`09b` оставляем как промежуточный эксперимент: он показывает, что если подгонять алгоритм слияния под ответ, оценивая параметры напрямую на golden, можно прийти к отличным результатам. Но это фактически утечка, поэтому дальше параметры выбираем по silver.

In [33]:
config_09b = AttachClusteringConfig(
    min_similarity=0.78,
    max_days=14,
    min_margin=0.03,
    source_max_cluster_size=2,
    require_evidence=True,
    title_jaccard_threshold=0.10,
    min_shared_numbers=1,
    cluster_prefix="exp09b",
)

cluster_ids_09b, diag_09b, attachments_09b = build_best_candidate_attach_clusters(
    old_pipeline_clustered_news,
    candidate_pairs,
    baseline_cluster_ids,
    config_09b,
)

metrics_cluster_00b = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    baseline_cluster_ids,
)
metrics_cluster_09b = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    cluster_ids_09b,
)

pd.DataFrame(
    [
        {"experiment": "exp_00b", **metrics_cluster_00b},
        {"experiment": "exp_09b_diagnostic", **metrics_cluster_09b},
    ]
)[
    [
        "experiment",
        "tp_same_pairs",
        "fp_false_merge_pairs",
        "fn_missed_same_pairs",
        "pairwise_precision",
        "pairwise_recall",
        "pairwise_f1",
        "false_merge_rate",
    ]
]

,experiment,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate
0,exp_00b,117,0,70,1.0,0.625668,0.769737,0.0
1,exp_09b_diagnostic,154,0,33,1.0,0.823529,0.903226,0.0


### 5.2. `exp_10`: выбор параметров слияния кластеров по silver-разметке

Исходя из предыдущего анализа, silver-разметка сильно фрагментирована: сильнее, чем ручная разметка. Поэтому надёжным weak-positive сигналом считаем только те случаи, когда даже в silver несколько новостей попали в один кластер. По этому сигналу подбираем параметры слияния, а затем проверяем результат на golden.

In [34]:
silver_for_clustering = prepare_silver_positive_reference(
    silver,
    golden,
    old_pipeline_clustered_news,
)

silver_cluster_sizes = silver_for_clustering["cluster_id"].value_counts()
print("Silver rows in candidate pool:", len(silver_for_clustering))
print("Silver clusters:", silver_for_clustering["cluster_id"].nunique())
print(
    "Silver positive pairs:",
    int(silver_cluster_sizes.map(lambda x: x * (x - 1) // 2).sum()),
)
print("Silver singleton share:", float((silver_cluster_sizes == 1).mean()))

selection_config = SilverPositiveSelectionConfig(
    max_pred_pair_growth_over_baseline=1.55,
    max_all_data_cluster_size=80,
)

(
    exp10_sweep,
    cluster_ids_by_variant_exp10,
    attachments_by_variant_exp10,
) = run_silver_positive_attach_sweep(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
    silver_reference=silver_for_clustering,
    base_cluster_ids=baseline_cluster_ids,
    selection_config=selection_config,
    candidate_pairs=candidate_pairs,
)

best_exp10_row = select_silver_positive_variant(exp10_sweep, selection_config)
best_exp10_variant = str(best_exp10_row["variant"])
best_exp10_cluster_ids = cluster_ids_by_variant_exp10[best_exp10_variant]
best_exp10_config = get_attach_config_from_sweep_row(best_exp10_row)
attachments_exp10 = attachments_by_variant_exp10.get(best_exp10_variant, pd.DataFrame())

print("Selected exp_10 variant:", best_exp10_variant)
print("Silver-positive recall:", best_exp10_row["silver_positive_recall"])
print("Silver predicted-pair growth:", best_exp10_row["silver_pred_pair_growth"])
print("Selected attachments:", len(attachments_exp10))

exp10_sweep.sort_values(
    ["silver_positive_recall", "silver_pred_pair_growth"],
    ascending=[False, True],
).head(20)

Silver rows in candidate pool: 1915
Silver clusters: 1736
Silver positive pairs: 242
Silver singleton share: 0.9141705069124424
Selected exp_10 variant: exp10_src2_sim0.75_days7_m0.03_tj0.15_num1
Silver-positive recall: 0.7975206611570248
Silver predicted-pair growth: 1.5255198487712665
Selected attachments: 150


,variant,min_similarity,max_days,min_margin,source_max_cluster_size,title_jaccard_threshold,min_shared_numbers,require_evidence,candidate_attach_edges,attached_source_clusters,attached_rows,all_data_max_cluster_size,silver_positive_pairs,silver_recovered_positive_pairs,silver_missed_positive_pairs,silver_positive_recall,silver_total_pred_pairs,silver_pred_pair_growth,cluster_prefix,candidate_attach_targets
7,exp10_src2_sim0.75_days7_m0.03_tj0.05_num1,0.75,7.0,0.03,2.0,0.05,1.0,True,443,179,194,35,242,194,48,0.801653,1793,1.694707,exp10_src2_sim0.75_days7_m0.03_tj0.05_num1,229.0
11,exp10_src2_sim0.75_days7_m0.03_tj0.15_num1,0.75,7.0,0.03,2.0,0.15,1.0,True,323,150,165,33,242,193,49,0.797521,1614,1.525520,exp10_src2_sim0.75_days7_m0.03_tj0.15_num1,179.0
9,exp10_src2_sim0.75_days7_m0.03_tj0.10_num1,0.75,7.0,0.03,2.0,0.10,1.0,True,353,163,178,35,242,193,49,0.797521,1655,1.564272,exp10_src2_sim0.75_days7_m0.03_tj0.10_num1,198.0
56,exp10_src2_sim0.75_days10_m0.03_tj0.05_num2,0.75,10.0,0.03,2.0,0.05,2.0,True,439,183,200,35,242,193,49,0.797521,1798,1.699433,exp10_src2_sim0.75_days10_m0.03_tj0.05_num2,242.0
55,exp10_src2_sim0.75_days10_m0.03_tj0.05_num1,0.75,10.0,0.03,2.0,0.05,1.0,True,497,193,210,35,242,193,49,0.797521,1832,1.731569,exp10_src2_sim0.75_days10_m0.03_tj0.05_num1,255.0
104,exp10_src2_sim0.75_days14_m0.03_tj0.05_num2,0.75,14.0,0.03,2.0,0.05,2.0,True,500,200,217,36,242,193,49,0.797521,1897,1.793006,exp10_src2_sim0.75_days14_m0.03_tj0.05_num2,267.0
59,exp10_src2_sim0.75_days10_m0.03_tj0.15_num1,0.75,10.0,0.03,2.0,0.15,1.0,True,362,163,180,34,242,192,50,0.793388,1668,1.576560,exp10_src2_sim0.75_days10_m0.03_tj0.15_num1,200.0
57,exp10_src2_sim0.75_days10_m0.03_tj0.10_num1,0.75,10.0,0.03,2.0,0.10,1.0,True,394,176,193,35,242,192,50,0.793388,1690,1.597353,exp10_src2_sim0.75_days10_m0.03_tj0.10_num1,220.0
200,exp10_src2_sim0.76_days10_m0.03_tj0.05_num2,0.76,10.0,0.03,2.0,0.05,2.0,True,336,154,167,35,242,192,50,0.793388,1710,1.616257,exp10_src2_sim0.76_days10_m0.03_tj0.05_num2,191.0
199,exp10_src2_sim0.76_days10_m0.03_tj0.05_num1,0.76,10.0,0.03,2.0,0.05,1.0,True,381,162,176,35,242,192,50,0.793388,1727,1.632325,exp10_src2_sim0.76_days10_m0.03_tj0.05_num1,201.0


In [35]:
exp10_clustered_news = make_clustered_news(old_pipeline_clustered_news, best_exp10_cluster_ids)

metrics_cluster_10 = evaluate_cluster_ids_on_reference(
    golden,
    old_pipeline_clustered_news,
    best_exp10_cluster_ids,
)

cluster_comparison = pd.DataFrame(
    [
        {"experiment": "exp_00b_baseline", **metrics_cluster_00b},
        {"experiment": "exp_09b_diagnostic", **metrics_cluster_09b},
        {"experiment": "exp_10_silver_positive", **metrics_cluster_10},
    ]
)

cluster_comparison[
    [
        "experiment",
        "tp_same_pairs",
        "fp_false_merge_pairs",
        "fn_missed_same_pairs",
        "pairwise_precision",
        "pairwise_recall",
        "pairwise_f1",
        "false_merge_rate",
    ]
]

,experiment,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate
0,exp_00b_baseline,117,0,70,1.000000,0.625668,0.769737,0.000000
1,exp_09b_diagnostic,154,0,33,1.000000,0.823529,0.903226,0.000000
2,exp_10_silver_positive,156,3,31,0.981132,0.834225,0.901734,0.018868


### Вывод
Метод оптимизации слияния прямо на `golden`, конечно, даёт хорошие результаты, но это некорректная подгонка под ответ. Аналогичный подход на silver-разметке корректнее: он даёт чуть более слабый результат и небольшой false merge rate около 0.0188, но в целом precision почти не страдает, а F1 заметно лучше оригинального алгоритма. Поэтому этот вариант выбран для дальнейших экспериментов.

## 6. Сравнение novelty classifiers на выбранной кластеризации

Кластеризация теперь фиксирована: `exp_10`. Дальше экспериментируем с моделью классификации. Сначала оцениваем, как ранее подготовленная референсная модель работает на новой кластеризации.

In [36]:
exp10_features = build_legacy_significance_features(
    news_df=exp10_clustered_news,
    embeddings=old_pipeline_embeddings,
)
exp10_features["news_id"] = normalize_news_id(exp10_features["news_id"])

train_frame = make_significance_training_frame(
    features_df=exp10_features,
    labels_df=silver,
    eval_df=golden,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
)

train_part, val_part = train_validation_split(
    train_frame,
    validation_size=0.25,
    random_state=42,
    group_column="cluster_id",
)

print("train_frame:", train_frame.shape)
print("train:", train_part.shape, "validation:", val_part.shape)
print("target distribution:")
print(train_frame["is_significant"].value_counts())

train_frame: (179, 24)
train: (140, 24) validation: (39, 24)
target distribution:
is_significant
1    133
0     46
Name: count, dtype: int64


In [37]:
def evaluate_exp10_model(experiment: str, model_wrapper, novelty_variant: str, comment: str):
    pred = model_wrapper.predict_clustered_with_fallback(
        news_df=exp10_clustered_news,
        embeddings=old_pipeline_embeddings,
    )
    metrics = tracker.register(
        experiment=experiment,
        golden=golden,
        prediction=pred,
        embedding_variant="BAAI/bge-m3 id-aligned",
        clustering=f"exp_10 silver-positive attach ({best_exp10_variant})",
        novelty_variant=novelty_variant,
        comment=comment,
    )
    return pred, metrics


# Контроль: текущая сохранённая модель на exp_10 кластеризации.
pred_10a, metrics_10a = evaluate_exp10_model(
    "exp_10a_current_model_on_exp10_clustering",
    current_model,
    "current CatBoost/fallback model",
    "Текущая сохранённая модель применена к выбранной exp_10 кластеризации",
)

tracker.compact().tail(3)

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Воспроизводимый референс из предыдущего этапа
1,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...


### Вывод
Прирост есть, но не очень большой. При этом исходные метрики модели уже были достаточно высокими, а основные важные характеристики последнего шага — `significant_f1` и recall — увеличились.

### 6.0. Подготовка новых моделей и контракта признаков

Модель классификации не должна видеть будущие новости из кластера. Для каждой новости признаки считаются только по предыдущим элементам того же предсказанного кластера. Это важно для будущего production-сценария: порядок обработки совпадает с хронологическим потоком новостей, а качество не завышается за счёт утечек из будущего.

In [38]:
feature_contract = pd.DataFrame(
    [
        {
            "feature_group": "cluster position",
            "features": "position_in_cluster, cluster_size_so_far",
            "signal": "является ли новость первой/ранней в сюжете",
        },
        {
            "feature_group": "time context",
            "features": "days_since_previous, days_since_cluster_start",
            "signal": "пауза с прошлого сообщения и возраст сюжета",
        },
        {
            "feature_group": "semantic similarity",
            "features": "max/mean/min/top-k/last_prev_similarity, previous_centroid_similarity",
            "signal": "насколько текущий текст похож на уже известные сообщения сюжета",
        },
        {
            "feature_group": "lexical evidence",
            "features": "title_jaccard_max, text_jaccard_max, shared_numbers_count, new_numbers_count",
            "signal": "совпадение формулировок и числовых фактов",
        },
        {
            "feature_group": "document shape",
            "features": "title_length, text_length",
            "signal": "косвенная полнота и тип материала",
        },
    ]
)

feature_contract

,feature_group,features,signal
0,cluster position,"position_in_cluster, cluster_size_so_far",является ли новость первой/ранней в сюжете
1,time context,"days_since_previous, days_since_cluster_start",пауза с прошлого сообщения и возраст сюжета
2,semantic similarity,"max/mean/min/top-k/last_prev_similarity, previ...",насколько текущий текст похож на уже известные...
3,lexical evidence,"title_jaccard_max, text_jaccard_max, shared_nu...",совпадение формулировок и числовых фактов
4,document shape,"title_length, text_length",косвенная полнота и тип материала


### 6.1. Обучение новых моделей CatBoost / MLP / LogisticRegression

Все модели обучаются на признаках и silver set, построенных поверх уже выбранной `exp_10`-кластеризации. Golden исключён из обучения и используется только для оценки.

Архитектурное улучшение относительно baseline состоит не только в замене классификатора. Пайплайн разделён на два независимых слоя: сначала слияние кластеров улучшает контекст сюжета, затем общая обёртка применяет одну из моделей классификации и одинаковую логику постобработки. Это позволяет сравнивать CatBoost, MLP и LogisticRegression честно: кластеризация и признаки одинаковые, меняется только классификатор и подобранный threshold.

Сначала берём валидационную часть silver и подбираем удачные thresholds, после чего обучаем финальные модели на полном train frame.

In [39]:
catboost_exp10_model, threshold_catboost = fit_catboost_binary(
    train_part,
    val_part,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    random_state=42,
)
pred_10b, metrics_10b = evaluate_exp10_model(
    "exp_10b_catboost_trained_on_exp10_clustering",
    catboost_exp10_model,
    "CatBoost trained on exp_10 features + fallback",
    "CatBoost обучен на silver, поверх выбранной exp_10 кластеризации",
)

threshold_catboost

0:	learn: 0.9116279	test: 0.9275362	best: 0.9275362 (0)	total: 1.45ms	remaining: 723ms
100:	learn: 0.9615385	test: 0.9253731	best: 0.9565217 (3)	total: 114ms	remaining: 449ms
200:	learn: 0.9900990	test: 0.9393939	best: 0.9565217 (3)	total: 220ms	remaining: 328ms
300:	learn: 1.0000000	test: 0.9393939	best: 0.9565217 (3)	total: 331ms	remaining: 219ms
400:	learn: 1.0000000	test: 0.9538462	best: 0.9565217 (3)	total: 446ms	remaining: 110ms
499:	learn: 1.0000000	test: 0.9538462	best: 0.9565217 (3)	total: 554ms	remaining: 0us

bestTest = 0.9565217391
bestIteration = 3

Shrink model to first 4 iterations.
0:	learn: 0.9175627	total: 1.4ms	remaining: 696ms
100:	learn: 0.9637681	total: 111ms	remaining: 439ms
200:	learn: 0.9815498	total: 224ms	remaining: 333ms
300:	learn: 0.9962547	total: 340ms	remaining: 225ms
400:	learn: 1.0000000	total: 458ms	remaining: 113ms
499:	learn: 1.0000000	total: 578ms	remaining: 0us


ThresholdSearchResult(threshold=0.49499999999999994, f1=0.9565217391304348, precision=0.9166666666666666, recall=1.0)

In [40]:
mlp_exp10_model, threshold_mlp = fit_mlp_binary(
    train_part,
    val_part,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    random_state=42,
)
pred_10c, metrics_10c = evaluate_exp10_model(
    "exp_10c_mlp_trained_on_exp10_clustering",
    mlp_exp10_model,
    "MLP trained on exp_10 features + fallback",
    "MLP обучена на silver features, построенных поверх выбранной exp_10 кластеризации",
)

threshold_mlp

ThresholdSearchResult(threshold=0.4499999999999999, f1=0.9428571428571428, precision=0.8918918918918919, recall=1.0)

In [41]:
logreg_exp10_model, threshold_logreg = fit_logreg_binary(
    train_part,
    val_part,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    random_state=42,
)
pred_10d, metrics_10d = evaluate_exp10_model(
    "exp_10d_logreg_trained_on_exp10_clustering",
    logreg_exp10_model,
    "LogisticRegression trained on exp_10 features + fallback",
    ("LogisticRegression trained on silver features built on top of selected exp_10 clustering"),
)

threshold_logreg

ThresholdSearchResult(threshold=0.195, f1=0.9565217391304348, precision=0.9166666666666666, recall=1.0)

### 6.2. Сравнение полученных threshold

Для каждой обученной модели threshold выбирается на валидационной выборке по F1 класса `significant`. После выбора порога модель переобучается на полном наборе silver, а golden остаётся только для финальной оценки.

In [42]:
threshold_table = pd.DataFrame(
    [
        {
            "experiment": "exp_10b_catboost_trained_on_exp10_clustering",
            "model": "CatBoostClassifier",
            "validation_threshold": threshold_catboost.threshold,
            "validation_precision": threshold_catboost.precision,
            "validation_recall": threshold_catboost.recall,
            "validation_f1": threshold_catboost.f1,
        },
        {
            "experiment": "exp_10c_mlp_trained_on_exp10_clustering",
            "model": "StandardScaler + MLPClassifier(64, 32)",
            "validation_threshold": threshold_mlp.threshold,
            "validation_precision": threshold_mlp.precision,
            "validation_recall": threshold_mlp.recall,
            "validation_f1": threshold_mlp.f1,
        },
        {
            "experiment": "exp_10d_logreg_trained_on_exp10_clustering",
            "model": "StandardScaler + balanced LogisticRegression",
            "validation_threshold": threshold_logreg.threshold,
            "validation_precision": threshold_logreg.precision,
            "validation_recall": threshold_logreg.recall,
            "validation_f1": threshold_logreg.f1,
        },
    ]
)

threshold_table.sort_values("validation_f1", ascending=False)

,experiment,model,validation_threshold,validation_precision,validation_recall,validation_f1
0,exp_10b_catboost_trained_on_exp10_clustering,CatBoostClassifier,0.495,0.916667,1.0,0.956522
2,exp_10d_logreg_trained_on_exp10_clustering,StandardScaler + balanced LogisticRegression,0.195,0.916667,1.0,0.956522
1,exp_10c_mlp_trained_on_exp10_clustering,"StandardScaler + MLPClassifier(64, 32)",0.450,0.891892,1.0,0.942857


## 7. Выбор и экспорт финальной модели

Выбираем лучшую модель среди `exp_10*` по `significant_f1`, при равенстве — по `significant_recall`. При желании можно вручную переопределить `SELECTED_MODEL_EXPERIMENT`.

In [43]:
model_selection_table = tracker.compact().copy()
model_selection_table = (
    model_selection_table[model_selection_table["experiment"].str.startswith("exp_10", na=False)]
    .sort_values(
        ["significant_f1", "significant_recall", "significant_precision"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

model_selection_table

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...
1,exp_10c_mlp_trained_on_exp10_clustering,0.901734,0.018868,0.747126,0.831169,0.876712,0.853333,"MLP обучена на silver features, построенных по..."
2,exp_10d_logreg_trained_on_exp10_clustering,0.901734,0.018868,0.712644,0.833333,0.821918,0.827586,LogisticRegression trained on silver features ...
3,exp_10b_catboost_trained_on_exp10_clustering,0.901734,0.018868,0.689655,0.848485,0.767123,0.805755,"CatBoost обучен на silver, поверх выбранной ex..."


In [44]:
SELECTED_MODEL_EXPERIMENT = model_selection_table.iloc[0]["experiment"]
print("Selected final model experiment:", SELECTED_MODEL_EXPERIMENT)

selected_model_objects = {
    "exp_10a_current_model_on_exp10_clustering": current_model,
    "exp_10b_catboost_trained_on_exp10_clustering": catboost_exp10_model,
    "exp_10c_mlp_trained_on_exp10_clustering": mlp_exp10_model,
    "exp_10d_logreg_trained_on_exp10_clustering": logreg_exp10_model,
}

selected_model = selected_model_objects[SELECTED_MODEL_EXPERIMENT]

final_model_dir = paths.models_dir / "final_exp10"
final_model_dir.mkdir(parents=True, exist_ok=True)
final_model_path = final_model_dir / f"{SELECTED_MODEL_EXPERIMENT}.joblib"
joblib.dump(selected_model, final_model_path)

# Save final pipeline config with selected exp_10 parameters.
final_config = FinalPipelineConfig(attach_clustering=best_exp10_config)
final_config_path = final_model_dir / "final_pipeline_config.json"
final_config.to_json(final_config_path)

print("Saved model:", final_model_path)
print("Saved pipeline config:", final_config_path)

Selected final model experiment: exp_10a_current_model_on_exp10_clustering
Saved model: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\final_exp10\exp_10a_current_model_on_exp10_clustering.joblib
Saved pipeline config: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\final_exp10\final_pipeline_config.json


### Выводы
Выбрана модель `exp_10a` как модель с наибольшим F1/recall. Это уже сохранённая ранее обученная модель. Второй претендент — MLP — имеет чуть худший recall, но тоже подходит как кандидат в финальные модели.

### 7.1. Постпроцессинг

Финальная модель сохраняется не как голый классификатор, а как обёртка над ним с постпроцессингом. Поэтому результаты используют одинаковые правила: вероятность `p_significant` превращается в `novelty_label`, почти-дубликаты получают label `duplicate`, пограничные случаи помечаются `needs_review`, а первые элементы кластера проходят fallback-поиск по предыдущим новостям той же темы.

In [45]:
postprocessing_contract = pd.DataFrame(
    [
        {
            "step": "binary threshold",
            "rule": "p_significant >= selected_model.config.threshold -> significant, иначе minor",
            "purpose": "контролируемый trade-off precision/recall",
        },
        {
            "step": "duplicate override",
            "rule": "если объект не significant и max_prev_similarity >= duplicate_threshold -> duplicate",
            "purpose": "отделить почти дословные повторы от обычных minor updates",
        },
        {
            "step": "review margin",
            "rule": "abs(p_significant - threshold) <= review_margin -> needs_review",
            "purpose": "выделить наиболее неопределённые случаи для ручной проверки",
        },
        {
            "step": "first-item fallback",
            "rule": "для первого элемента кластера ищем похожие более ранние новости той же темы в fallback window",
            "purpose": "не считать каждую первую новость нового predicted cluster автоматически значимой",
        },
        {
            "step": "pipeline config export",
            "rule": "final_pipeline_config.json хранит выбранные exp_10 attach-параметры",
            "purpose": "воспроизводимый inference вне ноутбука",
        },
    ]
)

display(postprocessing_contract)
print("Selected threshold:", selected_model.config.threshold)
print("Duplicate threshold:", selected_model.config.duplicate_threshold)
print("Review margin:", selected_model.config.review_margin)
print("Final pipeline config:", final_config_path)

,step,rule,purpose
0,binary threshold,p_significant >= selected_model.config.thresho...,контролируемый trade-off precision/recall
1,duplicate override,если объект не significant и max_prev_similari...,отделить почти дословные повторы от обычных mi...
2,review margin,abs(p_significant - threshold) <= review_margi...,выделить наиболее неопределённые случаи для ру...
3,first-item fallback,для первого элемента кластера ищем похожие бол...,не считать каждую первую новость нового predic...
4,pipeline config export,final_pipeline_config.json хранит выбранные ex...,воспроизводимый inference вне ноутбука


Selected threshold: 0.42
Duplicate threshold: 0.9
Review margin: 0.1
Final pipeline config: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\final_exp10\final_pipeline_config.json


## 8. Опциональная проверка дообученного BGE-M3

Эксперимент по применению дообученной (в ноутбуке `07`) модели bge_m3 на русских парафразах.

In [46]:
FINETUNED_BGE_MODEL_PATH = paths.models_dir / "news-flow-bge-m3-ru-paraphrase-mnrl" / "checkpoint-95"

FINETUNED_EMBEDDINGS_CACHE = paths.artifacts_dir / "embeddings" / "exp_08_finetuned_bge_m3_candidate_pool_embeddings.npz"

print("FINETUNED_BGE_MODEL_PATH:", FINETUNED_BGE_MODEL_PATH)
print("FINETUNED_EMBEDDINGS_CACHE:", FINETUNED_EMBEDDINGS_CACHE)

if FINETUNED_BGE_MODEL_PATH.exists():
    import gc

    if "encoder" in globals():
        del encoder
        gc.collect()

    finetuned_encoder = SentenceTransformerEncoder(
        model_name=str(FINETUNED_BGE_MODEL_PATH),
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    from model.attach_clustering import build_baseline_cluster_ids
    from model.embeddings import get_or_create_id_aligned_embeddings

    finetuned_embeddings = get_or_create_id_aligned_embeddings(
        encoder=finetuned_encoder,
        df=old_pipeline_clustered_news,
        cache_path=FINETUNED_EMBEDDINGS_CACHE,
        id_column="news_id",
        text_column="model_text",
    )
    finetuned_base_ids, _ = build_baseline_cluster_ids(
        old_pipeline_clustered_news,
        finetuned_embeddings,
        BaselineClusteringConfig(),
    )
    finetuned_pairs = build_candidate_pairs(
        old_pipeline_clustered_news,
        finetuned_embeddings,
        min_similarity=best_exp10_config.min_similarity,
        max_days=max(best_exp10_config.max_days, 14),
    )
    finetuned_ids, _, _ = build_best_candidate_attach_clusters(
        old_pipeline_clustered_news,
        finetuned_pairs,
        finetuned_base_ids,
        best_exp10_config,
    )
    finetuned_clustered_news = make_clustered_news(old_pipeline_clustered_news, finetuned_ids)
    pred_finetuned = selected_model.predict_clustered_with_fallback(
        news_df=finetuned_clustered_news,
        embeddings=finetuned_embeddings,
    )
    tracker.register(
        experiment="exp_11_finetuned_bge_with_selected_exp10_model",
        golden=golden,
        prediction=pred_finetuned,
        embedding_variant="fine-tuned BGE-M3",
        clustering="same exp_10 attach config",
        novelty_variant=SELECTED_MODEL_EXPERIMENT,
        comment="Ablation: проверка дообученного BGE-M3 без повторного подбора параметров",
    )
else:
    print(f"Дообученный BGE-M3 не найден: {FINETUNED_BGE_MODEL_PATH}")
    print("Ячейка пропущена. Укажите актуальный путь в FINETUNED_BGE_MODEL_PATH, если модель есть.")

FINETUNED_BGE_MODEL_PATH: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\news-flow-bge-m3-ru-paraphrase-mnrl\checkpoint-95
FINETUNED_EMBEDDINGS_CACHE: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\embeddings\exp_08_finetuned_bge_m3_candidate_pool_embeddings.npz


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Рёбер в графе похожести: 828
Количество кластеров: 2639


## 9. Финальное сравнение полученного результата с референсной моделью

In [47]:
final_results = tracker.compact().copy()
final_results.sort_values(
    ["significant_f1", "significant_recall", "pairwise_f1"],
    ascending=[False, False, False],
)

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
1,exp_10a_current_model_on_exp10_clustering,0.901734,0.018868,0.758621,0.842105,0.876712,0.859060,Текущая сохранённая модель применена к выбранн...
3,exp_10c_mlp_trained_on_exp10_clustering,0.901734,0.018868,0.747126,0.831169,0.876712,0.853333,"MLP обучена на silver features, построенных по..."
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Воспроизводимый референс из предыдущего этапа
4,exp_10d_logreg_trained_on_exp10_clustering,0.901734,0.018868,0.712644,0.833333,0.821918,0.827586,LogisticRegression trained on silver features ...
5,exp_11_finetuned_bge_with_selected_exp10_model,0.808777,0.022727,0.701149,0.821918,0.821918,0.821918,Ablation: проверка дообученного BGE-M3 без пов...
2,exp_10b_catboost_trained_on_exp10_clustering,0.901734,0.018868,0.689655,0.848485,0.767123,0.805755,"CatBoost обучен на silver, поверх выбранной ex..."


In [48]:
baseline_row = final_results[final_results["experiment"].eq("exp_00b_reproduced_old_pipeline")].iloc[0]
selected_row = final_results[final_results["experiment"].eq(SELECTED_MODEL_EXPERIMENT)].iloc[0]

compare_columns = [
    "pairwise_f1",
    "false_merge_rate",
    "significance_accuracy",
    "significant_precision",
    "significant_recall",
    "significant_f1",
]

baseline_vs_selected = pd.DataFrame(
    {
        "metric": compare_columns,
        "baseline_00b": [baseline_row[column] for column in compare_columns],
        "selected_exp10": [selected_row[column] for column in compare_columns],
    }
)
baseline_vs_selected["delta"] = (
    baseline_vs_selected["selected_exp10"] - baseline_vs_selected["baseline_00b"]
)
baseline_vs_selected

,metric,baseline_00b,selected_exp10,delta
0,pairwise_f1,0.769737,0.901734,0.131997
1,false_merge_rate,0.000000,0.018868,0.018868
2,significance_accuracy,0.724138,0.758621,0.034483
3,significant_precision,0.835616,0.842105,0.006489
4,significant_recall,0.835616,0.876712,0.041096
5,significant_f1,0.835616,0.859060,0.023444


### 9.1. Detailed quality breakdown

Основная таблица выше ранжирует эксперименты по итоговым F1. Для диагностики ошибок отдельно смотрим coverage, парные ошибки кластеризации и классификации. Это показывает, за счёт чего получен прирост: меньше пропущенных пар в одном сюжете и немного выше recall значимых новостей при контролируемом росте некорректных слияний.

In [49]:
quality_breakdown_columns = [
    "experiment",
    "coverage",
    "cluster_eval_items",
    "total_ref_pairs",
    "total_pred_pairs",
    "tp_same_pairs",
    "fp_false_merge_pairs",
    "fn_missed_pairs",
    "pairwise_precision",
    "pairwise_recall",
    "pairwise_f1",
    "novelty_eval_rows",
    "significant_tp",
    "significant_fp",
    "significant_fn",
    "significant_tn",
    "significant_precision",
    "significant_recall",
    "significant_f1",
]

quality_breakdown = (
    tracker.results_table
    .drop_duplicates(subset=["experiment"], keep="last")
    .loc[
        lambda df: df["experiment"].isin(
            ["exp_00b_reproduced_old_pipeline", SELECTED_MODEL_EXPERIMENT]
        )
    ]
    [quality_breakdown_columns]
    .reset_index(drop=True)
)

quality_breakdown

,experiment,coverage,cluster_eval_items,total_ref_pairs,total_pred_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_pairs,pairwise_precision,pairwise_recall,pairwise_f1,novelty_eval_rows,significant_tp,significant_fp,significant_fn,significant_tn,significant_precision,significant_recall,significant_f1
0,exp_00b_reproduced_old_pipeline,1.0,121,187,117,117,0,70,1.000000,0.625668,0.769737,87,61,12,12,2,0.835616,0.835616,0.835616
1,exp_10a_current_model_on_exp10_clustering,1.0,121,187,159,156,3,31,0.981132,0.834225,0.901734,87,64,12,9,2,0.842105,0.876712,0.859060


In [50]:
metric_interpretation = pd.DataFrame(
    [
        {
            "metric": "pairwise_precision",
            "what_it_checks": "доля предсказанных same-story пар, которые есть в golden",
            "failure_mode": "false merge: разные сюжеты ошибочно склеены",
        },
        {
            "metric": "pairwise_recall",
            "what_it_checks": "доля golden same-story пар, найденных кластеризацией",
            "failure_mode": "fragmentation: один сюжет разбит на несколько кластеров",
        },
        {
            "metric": "false_merge_rate",
            "what_it_checks": "FP / predicted same-story pairs",
            "failure_mode": "контроль риска слишком агрессивного attach",
        },
        {
            "metric": "significant_precision",
            "what_it_checks": "сколько предсказанных significant действительно significant",
            "failure_mode": "ложные сигналы важности",
        },
        {
            "metric": "significant_recall",
            "what_it_checks": "сколько golden significant найдено моделью",
            "failure_mode": "пропуск важных новостей",
        },
        {
            "metric": "significant_f1",
            "what_it_checks": "баланс precision/recall для главного binary target",
            "failure_mode": "итоговый компромисс novelty-классификации",
        },
    ]
)

metric_interpretation

,metric,what_it_checks,failure_mode
0,pairwise_precision,"доля предсказанных same-story пар, которые ест...",false merge: разные сюжеты ошибочно склеены
1,pairwise_recall,"доля golden same-story пар, найденных кластери...",fragmentation: один сюжет разбит на несколько ...
2,false_merge_rate,FP / predicted same-story pairs,контроль риска слишком агрессивного attach
3,significant_precision,сколько предсказанных significant действительн...,ложные сигналы важности
4,significant_recall,сколько golden significant найдено моделью,пропуск важных новостей
5,significant_f1,баланс precision/recall для главного binary ta...,итоговый компромисс novelty-классификации


## 10. Выводы

1. Baseline `exp_00b` был зафиксирован как воспроизводимая контрольная точка: BGE-M3, строгая графовая кластеризация с временным окном и сохранённая CatBoost-модель классификации.
2. Основное ограничение baseline — недосклейка сюжетов: высокая precision кластеризации достигается ценой низкого recall same-story пар.
3. Silver-разметка оказалась перефрагментированной и не подходит как полноценный источник отрицательных пар для разделения сюжетов. Поэтому в `exp_10` она использована только как weak-positive сигнал на склейку.
4. `exp_10` выбирает параметры кластеризации без использования golden-разметки, чтобы предотвратить утечку.
5. После фиксации `exp_10`-кластеризации отдельно сравниваются модели классификации (финального шага) по детекции новизны. Такой порядок разделяет две задачи: сначала кластеризацию, потом классификацию.
6. Финальная модель сохраняется в `data/artifacts/models/final_exp10/` и используется тем же кодом, что и финальный пайплайн.
7. По сохранённому запуску выбранная модель `exp_10` улучшает `pairwise_f1` (f1 кластеризации) с 0.7697 до 0.9017 и `significant_f1` (f1 классификации) с 0.8356 до 0.8591. Цена улучшения кластерного recall — небольшой рост `false_merge_rate` до 0.0189, что остаётся контролируемым для выбранного алгоритма.
8. Было проведено ещё несколько экспериментов, но их стоит лишь упомянуть: существенного прироста они не дали. Основной честный improvement — `exp_10` + выбранная модель классификации.

Итоговая архитектура:

- подготовка данных: единая выборка данных для обучения, silver set без утечек из golden, BGE-M3 в виде векторизатора, previous-only 18-feature contract;
- кластеризация: слияние кластеров позволило улучшить кластеризацию, за счёт этого выиграли и характеристики классификатора;
- классификация: эксперименты с переобучением моделей и использованием MLP не дали существенного прироста, поэтому остановились на уже обученной модели;
- постпроцессинг: после выдачи результатов классификатором, мы зафиксировали наиболее удачные пороги (threshold) для разметки и сделали экспорт `final_pipeline_config.json`;
- был произведён анализ качества: сравнение с референсом по метрикам кластеризации и классификации, оценены изменения, прирост и количество ошибок.